In [12]:
# ════════════════════════════════════════════════════════════════════════════════════
# PBT-V v2: Large-Scale Vision Experiment
# Predictive Boundary Theory — Ninthanawat N.
# Boundary Research Initiative, Bangkok, Thailand
#
# Model: DINOv2-Base (768 dim, 12 layers)
# Dataset: CIFAR-10 Temporal Sequences (2,000+ sequences, real images)
# Tests: Ablation (E/V/M), Noise Robustness, Tricky-Safe, Statistical Analysis
# Requirements: Colab Pro with A100 GPU
# ════════════════════════════════════════════════════════════════════════════════════

# ════════════════════════════════════════════
# CELL 0: Install
# ════════════════════════════════════════════
!pip install -q transformers accelerate datasets scikit-learn matplotlib seaborn torchvision


In [13]:
# ════════════════════════════════════════════
# CELL 1: Architecture — PBT-V v2 (DINOv2-Base)
# ════════════════════════════════════════════
import os, json, random, math, warnings, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from transformers import AutoImageProcessor, AutoModel
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torchvision
import torchvision.transforms as transforms
from PIL import Image
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
  print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1024**3:.0f} GB)')

print("\n" + "=" * 90)
print("  PBT-V v2: Large-Scale Vision Experiment")
print("  DINOv2-Base Backbone | CIFAR-10 Temporal Sequences")
print("  Predictive Boundary Theory — Ninthanawat N.")
print("=" * 90)

# ─── Load DINOv2-Base ───
MODEL_ID = "facebook/dinov2-base"
print(f"\n⏳ Loading {MODEL_ID}...")
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
base_model = AutoModel.from_pretrained(MODEL_ID).to(device)
base_model.eval()
for p in base_model.parameters():
    p.requires_grad_(False)

D_MODEL = 768      # DINOv2-Base hidden dim
N_LAYERS = 12      # 12 transformer layers
NUM_HEADS = 12     # 12 attention heads
D_K = D_MODEL // NUM_HEADS  # 64

print(f"✅ DINOv2-Base loaded")
print(f"\n📐 PBT Architecture Mapping:")
print(f"  C (d_model) = {D_MODEL}")
print(f"  σ (heads)   = {NUM_HEADS}")
print(f"  ρ (d_k)     = {D_K}")
print(f"  τ (layers)  = {N_LAYERS}")
print(f"  σ × ρ = {NUM_HEADS} × {D_K} = {NUM_HEADS * D_K} {'= C ✅' if NUM_HEADS*D_K==D_MODEL else ''}")

# ════════════════════════════════════════════
# PBT-V ADAPTER v2
# ════════════════════════════════════════════
class PBTVisionAdapterV2(nn.Module):
    """
    PBT-V v2: Full E→V→M pipeline for temporal vision processing.

    Module E: ε_l(t) = Π_l · (h_t(l) − h_{t-1}(l))   [temporal prediction error]
    Module V: valence_probes(ε_l) → [V_pain, V_pleasure, V_epistemic]
    Module M: V_acc(t) = v_step(t) + (1−γ) · V_acc(t−1)  [temporal EMA]
    """
    def __init__(self, d_model=D_MODEL, n_layers=N_LAYERS):
        super().__init__()
        self.d_model = d_model
        self.n_layers = n_layers

        # Module E: learnable precision per layer (log-space for positive guarantee)
        self.log_precision = nn.Parameter(torch.zeros(n_layers))

        # Module V: classify prediction error → 3D valence
        self.valence_probes = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, 256), nn.ReLU(), nn.Dropout(0.1),
                nn.Linear(256, 3), nn.Sigmoid()
            ) for _ in range(n_layers)
        ])

        # Module M: differential decay (γ_p < γ_pl < γ_e enforced via sort)
        self.raw_gamma = nn.Parameter(torch.tensor([-2.44, -1.92, -1.27]))

        # Classifier: [h_last | v_feat | v_acc] → safe/unsafe
        in_features = d_model + (n_layers * 3) + 3
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 512), nn.ReLU(), nn.Dropout(0.15),
            nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, 2)
        )

        # Internal state
        self.prev_h = None
        self.v_acc = None

    def get_precision(self):
        return torch.exp(self.log_precision)  # always positive

    def get_gamma(self):
        return torch.sigmoid(self.raw_gamma).sort().values  # enforced ordering

    def reset_state(self):
        self.prev_h = None
        self.v_acc = None

    def forward(self, h_current, return_valence=False):
        """
        h_current: [B, N_LAYERS, D_MODEL] - hidden states from current frame
        """
        B, L, D = h_current.shape

        if self.prev_h is None or self.prev_h.shape[0] != B:
            self.prev_h = h_current.detach()
            self.v_acc = torch.zeros(B, 3, device=h_current.device)

        # Module E: temporal prediction error
        precision = self.get_precision().view(1, -1, 1)  # [1, L, 1]
        epsilon = (h_current - self.prev_h) * precision   # [B, L, D]

        # Module V: classify each ε_l
        valences = []
        for l in range(self.n_layers):
            v_l = self.valence_probes[l](epsilon[:, l, :])  # [B, 3]
            valences.append(v_l)
        v_feat = torch.cat(valences, dim=1)        # [B, L*3]
        v_layers = torch.stack(valences, dim=1)    # [B, L, 3]
        v_step = v_layers.mean(dim=1)              # [B, 3]

        # Module M: temporal accumulation with differential decay
        gamma = self.get_gamma()
        self.v_acc = v_step + (1 - gamma) * self.v_acc

        # Classifier
        h_last = h_current[:, -1, :]  # [B, D]
        combined = torch.cat([h_last, v_feat, self.v_acc], dim=1)
        logits = self.classifier(combined)

        # Update prediction for next frame
        self.prev_h = h_current.detach()

        if return_valence:
            return logits, self.v_acc.detach(), v_layers.detach(), epsilon.detach()
        return logits, self.v_acc.detach(), v_feat.detach()

adapter = PBTVisionAdapterV2().to(device)
total_params = sum(p.numel() for p in adapter.parameters())
print(f"\n📐 PBT-V v2 Adapter: {total_params:,} trainable params")
print(f"  Module E: {sum(p.numel() for p in [adapter.log_precision]):,} (precision)")
print(f"  Module V: {sum(p.numel() for p in adapter.valence_probes.parameters()):,} (probes)")
print(f"  Module M: {sum(p.numel() for p in [adapter.raw_gamma]):,} (gamma)")
print(f"  Classifier: {sum(p.numel() for p in adapter.classifier.parameters()):,}")



Device: cuda
GPU: NVIDIA A100-SXM4-40GB (39 GB)

  PBT-V v2: Large-Scale Vision Experiment
  DINOv2-Base Backbone | CIFAR-10 Temporal Sequences
  Predictive Boundary Theory — Ninthanawat N.

⏳ Loading facebook/dinov2-base...


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

✅ DINOv2-Base loaded

📐 PBT Architecture Mapping:
  C (d_model) = 768
  σ (heads)   = 12
  ρ (d_k)     = 64
  τ (layers)  = 12
  σ × ρ = 12 × 64 = 768 = C ✅

📐 PBT-V v2 Adapter: 2,851,253 trainable params
  Module E: 12 (precision)
  Module V: 2,371,620 (probes)
  Module M: 3 (gamma)
  Classifier: 479,618


In [14]:
# ════════════════════════════════════════════
# CELL 2: Create CIFAR-10 Temporal Sequences
# ════════════════════════════════════════════
print("\n" + "=" * 90)
print("  📊 Creating CIFAR-10 Temporal Sequences")
print("=" * 90)

# Download CIFAR-10
print("⏳ Downloading CIFAR-10...")
cifar_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
cifar_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True)

# CIFAR-10 classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck
# PBT-V scenario: "robot driving on road"
#   Normal context: automobile(1), truck(9) = expected objects
#   Anomaly: airplane(0), ship(8), frog(6) = unexpected objects on road
#   Tricky-safe: bird(2), deer(4), dog(5) = unusual but not dangerous
NORMAL_CLASSES = [1, 9]       # automobile, truck
ANOMALY_CLASSES = [0, 8, 6]   # airplane, ship, frog (clearly wrong context)
TRICKY_CLASSES = [2, 4, 5]    # bird, deer, dog (unusual but safe)

# Sort images by class
images_by_class = {i: [] for i in range(10)}
for img, label in cifar_train:
    images_by_class[label].append(img)
for img, label in cifar_test:
    images_by_class[label].append(img)

print(f"  Images per class: { {k: len(v) for k, v in images_by_class.items()} }")

SEQ_LEN = 8  # 8 frames per sequence

def create_sequence(seq_type, seq_len=SEQ_LEN):
    """
    Create a temporal sequence of real CIFAR-10 images.
    Returns: (list_of_PIL_images, list_of_labels)

    normal:     8 frames of normal objects (all label=0 SAFE)
    anomaly:    4 normal + 4 anomaly (label=[0,0,0,0,1,1,1,1])
    tricky:     4 normal + 4 tricky-safe (all label=0 SAFE)
    """
    frames = []
    labels = []

    if seq_type == "normal":
        for _ in range(seq_len):
            cls = random.choice(NORMAL_CLASSES)
            img = random.choice(images_by_class[cls])
            frames.append(img)
            labels.append(0)

    elif seq_type == "anomaly":
        # First half: normal
        for _ in range(seq_len // 2):
            cls = random.choice(NORMAL_CLASSES)
            img = random.choice(images_by_class[cls])
            frames.append(img)
            labels.append(0)
        # Second half: anomaly appears
        for _ in range(seq_len - seq_len // 2):
            cls = random.choice(ANOMALY_CLASSES)
            img = random.choice(images_by_class[cls])
            frames.append(img)
            labels.append(1)

    elif seq_type == "tricky_safe":
        # First half: normal
        for _ in range(seq_len // 2):
            cls = random.choice(NORMAL_CLASSES)
            img = random.choice(images_by_class[cls])
            frames.append(img)
            labels.append(0)
        # Second half: unusual but safe
        for _ in range(seq_len - seq_len // 2):
            cls = random.choice(TRICKY_CLASSES)
            img = random.choice(images_by_class[cls])
            frames.append(img)
            labels.append(0)  # still SAFE

    return frames, labels

# Generate dataset
N_NORMAL = 800
N_ANOMALY = 800
N_TRICKY = 400

print(f"\n⏳ Generating sequences: {N_NORMAL} normal + {N_ANOMALY} anomaly + {N_TRICKY} tricky-safe")

all_sequences = []
for _ in range(N_NORMAL):
    frames, labels = create_sequence("normal")
    all_sequences.append({"frames": frames, "labels": labels, "type": "normal"})

for _ in range(N_ANOMALY):
    frames, labels = create_sequence("anomaly")
    all_sequences.append({"frames": frames, "labels": labels, "type": "anomaly"})

for _ in range(N_TRICKY):
    frames, labels = create_sequence("tricky_safe")
    all_sequences.append({"frames": frames, "labels": labels, "type": "tricky_safe"})

random.shuffle(all_sequences)

# Split: 70% train, 15% val, 15% test
n_total = len(all_sequences)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)
train_seqs = all_sequences[:n_train]
val_seqs = all_sequences[n_train:n_train+n_val]
test_seqs = all_sequences[n_train+n_val:]

print(f"\n  Total: {n_total} sequences × {SEQ_LEN} frames = {n_total * SEQ_LEN} frames")
print(f"  Train: {len(train_seqs)} | Val: {len(val_seqs)} | Test: {len(test_seqs)}")
print(f"  Types: normal={N_NORMAL}, anomaly={N_ANOMALY}, tricky_safe={N_TRICKY}")




  📊 Creating CIFAR-10 Temporal Sequences
⏳ Downloading CIFAR-10...
  Images per class: {0: 6000, 1: 6000, 2: 6000, 3: 6000, 4: 6000, 5: 6000, 6: 6000, 7: 6000, 8: 6000, 9: 6000}

⏳ Generating sequences: 800 normal + 800 anomaly + 400 tricky-safe

  Total: 2000 sequences × 8 frames = 16000 frames
  Train: 1400 | Val: 300 | Test: 300
  Types: normal=800, anomaly=800, tricky_safe=400


In [15]:
# ════════════════════════════════════════════
# CELL 3: Pre-compute Hidden States
# ════════════════════════════════════════════
print("\n" + "=" * 90)
print("  ⚡ Pre-computing Hidden States (DINOv2-Base)")
print("=" * 90)

def extract_sequence_features(sequences, desc=""):
    """Extract DINOv2 CLS tokens for all frames in all sequences."""
    all_h = []      # list of [SEQ_LEN, N_LAYERS, D_MODEL]
    all_labels = []  # list of [SEQ_LEN]
    all_types = []

    t0 = time.time()
    for i, seq in enumerate(sequences):
        seq_h = []
        for frame_img in seq["frames"]:
            inputs = processor(images=frame_img, return_tensors="pt").to(device)
            with torch.no_grad():
                outputs = base_model(**inputs, output_hidden_states=True)
            # CLS token from each layer
            layer_h = torch.stack([
                outputs.hidden_states[l+1][:, 0, :].squeeze(0).cpu()
                for l in range(N_LAYERS)
            ], dim=0)  # [N_LAYERS, D_MODEL]
            seq_h.append(layer_h)

        all_h.append(torch.stack(seq_h, dim=0))  # [SEQ_LEN, N_LAYERS, D_MODEL]
        all_labels.append(torch.tensor(seq["labels"]))
        all_types.append(seq["type"])

        if (i+1) % 200 == 0:
            elapsed = time.time() - t0
            print(f"    {desc} {i+1}/{len(sequences)} ({elapsed:.0f}s)")

    elapsed = time.time() - t0
    print(f"  ✅ {desc} done: {len(sequences)} sequences in {elapsed:.0f}s")
    return all_h, all_labels, all_types

print("\n  Pre-computing TRAIN...")
train_h, train_labels, train_types = extract_sequence_features(train_seqs, "Train")
print("\n  Pre-computing VAL...")
val_h, val_labels, val_types = extract_sequence_features(val_seqs, "Val")
print("\n  Pre-computing TEST...")
test_h, test_labels, test_types = extract_sequence_features(test_seqs, "Test")




  ⚡ Pre-computing Hidden States (DINOv2-Base)

  Pre-computing TRAIN...
    Train 200/1400 (16s)
    Train 400/1400 (32s)
    Train 600/1400 (48s)
    Train 800/1400 (64s)
    Train 1000/1400 (79s)
    Train 1200/1400 (95s)
    Train 1400/1400 (111s)
  ✅ Train done: 1400 sequences in 111s

  Pre-computing VAL...
    Val 200/300 (16s)
  ✅ Val done: 300 sequences in 24s

  Pre-computing TEST...
    Test 200/300 (16s)
  ✅ Test done: 300 sequences in 24s


In [16]:
# ════════════════════════════════════════════
# CELL 4: Training with BPTT
# ════════════════════════════════════════════
print("\n" + "=" * 90)
print("  🏋️ Training PBT-V v2 (BPTT on CIFAR-10 Sequences)")
print("=" * 90)

# Differential LR: Module E+M get higher LR
optimizer = AdamW([
    {'params': adapter.classifier.parameters(), 'lr': 1e-3},
    {'params': adapter.valence_probes.parameters(), 'lr': 1e-3},
    {'params': [adapter.log_precision], 'lr': 5e-3, 'weight_decay': 0},
    {'params': [adapter.raw_gamma], 'lr': 5e-3, 'weight_decay': 0},
])
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

EPOCHS = 30
best_val_acc = 0
best_state = None
history = {'loss': [], 'train_acc': [], 'val_acc': [],
           'gamma_p': [], 'gamma_pl': [], 'gamma_e': [],
           'precision_mean': [], 'precision_max': []}

adapter.train()
t0 = time.time()

for epoch in range(EPOCHS):
    epoch_loss = 0
    all_preds, all_true = [], []

    indices = list(range(len(train_h)))
    random.shuffle(indices)

    for idx in indices:
        seq_h = train_h[idx].to(device)      # [SEQ_LEN, N_LAYERS, D_MODEL]
        seq_labels = train_labels[idx].to(device)  # [SEQ_LEN]

        adapter.reset_state()
        optimizer.zero_grad()

        seq_loss = 0
        for t in range(SEQ_LEN):
            h_t = seq_h[t].unsqueeze(0)       # [1, N_LAYERS, D_MODEL]
            label_t = seq_labels[t].unsqueeze(0)

            logits, v_acc, v_feat = adapter(h_t)
            loss = criterion(logits, label_t)
            seq_loss = seq_loss + loss

            pred = logits.argmax(-1).item()
            all_preds.append(pred)
            all_true.append(label_t.item())

        seq_loss.backward()
        torch.nn.utils.clip_grad_norm_(adapter.parameters(), 1.0)
        optimizer.step()
        epoch_loss += seq_loss.item()

    scheduler.step()

    # Validation
    adapter.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for idx in range(len(val_h)):
            adapter.reset_state()
            for t in range(SEQ_LEN):
                h_t = val_h[idx][t].unsqueeze(0).to(device)
                logits, _, _ = adapter(h_t)
                val_preds.append(logits.argmax(-1).item())
                val_true.append(val_labels[idx][t].item())
    adapter.train()

    train_acc = accuracy_score(all_true, all_preds)
    val_acc = accuracy_score(val_true, val_preds)
    avg_loss = epoch_loss / len(train_h)

    gammas = adapter.get_gamma().detach().cpu().numpy()
    precs = adapter.get_precision().detach().cpu().numpy()

    history['loss'].append(avg_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['gamma_p'].append(gammas[0])
    history['gamma_pl'].append(gammas[1])
    history['gamma_e'].append(gammas[2])
    history['precision_mean'].append(precs.mean())
    history['precision_max'].append(precs.max())

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(adapter.state_dict())

    elapsed = time.time() - t0
    if (epoch+1) % 5 == 0:
        print(f"  Epoch {epoch+1:2d}/{EPOCHS}: Loss={avg_loss:.4f} Train={train_acc:.1%} Val={val_acc:.1%} "
              f"γ_p={gammas[0]:.4f} γ_e={gammas[2]:.4f} Π_max={precs.max():.4f} ({elapsed:.0f}s)")

adapter.load_state_dict(best_state)
adapter.eval()
print(f"\n✅ Best val accuracy: {best_val_acc:.1%}")

# Save
torch.save({"state_dict": adapter.state_dict(), "config": {
    "d_model": D_MODEL, "n_layers": N_LAYERS, "best_val_acc": best_val_acc,
    "n_train": len(train_h), "n_val": len(val_h), "n_test": len(test_h)
}}, "pbt_vision_v2.pth")

# Plot training
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0,0].plot(history['loss'], 'b-', lw=2); axes[0,0].set_title('Loss'); axes[0,0].grid(True, alpha=0.3)
axes[0,1].plot(history['train_acc'], 'g-', lw=2, label='Train')
axes[0,1].plot(history['val_acc'], 'r-', lw=2, label='Val')
axes[0,1].legend(); axes[0,1].set_title('Accuracy'); axes[0,1].grid(True, alpha=0.3)
axes[1,0].plot(history['gamma_p'], 'r-', lw=2, label='γ_pain')
axes[1,0].plot(history['gamma_pl'], 'g-', lw=2, label='γ_pleasure')
axes[1,0].plot(history['gamma_e'], 'b-', lw=2, label='γ_epistemic')
axes[1,0].legend(); axes[1,0].set_title('Module M: Gamma (Decay Rates)'); axes[1,0].grid(True, alpha=0.3)
axes[1,1].plot(history['precision_mean'], 'purple', lw=2, label='Π mean')
axes[1,1].plot(history['precision_max'], 'orange', lw=2, label='Π max')
axes[1,1].axhline(1.0, color='gray', ls='--'); axes[1,1].legend()
axes[1,1].set_title('Module E: Precision Π_l'); axes[1,1].grid(True, alpha=0.3)
plt.suptitle(f'PBT-V v2 Training ({len(train_h)} sequences) — Ninthanawat N.', fontweight='bold')
plt.tight_layout(); plt.savefig('pbtv2_training.png', dpi=150); plt.show()




  🏋️ Training PBT-V v2 (BPTT on CIFAR-10 Sequences)
  Epoch  5/30: Loss=0.0262 Train=99.9% Val=99.8% γ_p=0.0875 γ_e=0.5031 Π_max=2.5243 (547s)
  Epoch 10/30: Loss=0.0000 Train=100.0% Val=99.9% γ_p=0.1049 γ_e=0.5672 Π_max=3.0320 (1092s)
  Epoch 15/30: Loss=0.0000 Train=100.0% Val=99.9% γ_p=0.1048 γ_e=0.5675 Π_max=3.0310 (1635s)
  Epoch 20/30: Loss=0.0000 Train=100.0% Val=99.8% γ_p=0.1035 γ_e=0.5709 Π_max=3.1201 (2171s)
  Epoch 25/30: Loss=0.0000 Train=100.0% Val=99.9% γ_p=0.1030 γ_e=0.5717 Π_max=3.0336 (2707s)
  Epoch 30/30: Loss=0.0000 Train=100.0% Val=99.9% γ_p=0.1039 γ_e=0.5694 Π_max=3.0310 (3243s)

✅ Best val accuracy: 99.9%


In [17]:
# ════════════════════════════════════════════
# CELL 5: Comprehensive Test
# ════════════════════════════════════════════
print("\n" + "=" * 90)
print("  🧪 Comprehensive Test Results — PBT-V v2")
print("=" * 90)

adapter.eval()
test_preds, test_true, test_type_list = [], [], []

with torch.no_grad():
    for idx in range(len(test_h)):
        adapter.reset_state()
        for t in range(SEQ_LEN):
            h_t = test_h[idx][t].unsqueeze(0).to(device)
            logits, _, _ = adapter(h_t)
            test_preds.append(logits.argmax(-1).item())
            test_true.append(test_labels[idx][t].item())
            test_type_list.append(test_types[idx])

overall_acc = accuracy_score(test_true, test_preds)
print(f"\n📊 Overall Accuracy: {overall_acc:.1%} ({sum(1 for p,t in zip(test_preds,test_true) if p==t)}/{len(test_true)})")
print(f"\n{classification_report(test_true, test_preds, target_names=['SAFE','ANOMALY'], digits=4)}")

# Per-type
print("📊 By Sequence Type:")
for stype in ["normal", "anomaly", "tricky_safe"]:
    idx = [i for i, t in enumerate(test_type_list) if t == stype]
    if idx:
        acc = accuracy_score([test_true[i] for i in idx], [test_preds[i] for i in idx])
        print(f"  {stype:15s}: {acc:.1%} ({len(idx)} frames)")

# Confusion matrix
cm = confusion_matrix(test_true, test_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=["SAFE", "ANOMALY"], yticklabels=["SAFE", "ANOMALY"])
ax.set_title(f'PBT-V v2 — Acc: {overall_acc:.1%} ({len(test_h)} sequences)')
plt.tight_layout(); plt.savefig('pbtv2_confusion.png', dpi=150); plt.show()




  🧪 Comprehensive Test Results — PBT-V v2

📊 Overall Accuracy: 99.8% (2395/2400)

              precision    recall  f1-score   support

        SAFE     0.9995    0.9979    0.9987      1944
     ANOMALY     0.9913    0.9978    0.9945       456

    accuracy                         0.9979      2400
   macro avg     0.9954    0.9979    0.9966      2400
weighted avg     0.9979    0.9979    0.9979      2400

📊 By Sequence Type:
  normal         : 99.9% (888 frames)
  anomaly        : 99.9% (912 frames)
  tricky_safe    : 99.5% (600 frames)


In [18]:
# ════════════════════════════════════════════
# CELL 6: ABLATION TEST — E vs V vs M
# ════════════════════════════════════════════
print("\n" + "=" * 90)
print("  🔬 Ablation Test: Which Modules Matter?")
print("=" * 90)

def evaluate_ablation(adapter_state, ablate_E=False, ablate_M=False, name=""):
    """Test with specific modules disabled."""
    abl_adapter = PBTVisionAdapterV2().to(device)
    abl_adapter.load_state_dict(adapter_state)
    abl_adapter.eval()

    # Ablate Module E: set precision to 1.0 (neutral)
    if ablate_E:
        with torch.no_grad():
            abl_adapter.log_precision.fill_(0.0)  # exp(0)=1

    preds, true = [], []
    with torch.no_grad():
        for idx in range(len(test_h)):
            abl_adapter.reset_state()

            for t in range(SEQ_LEN):
                h_t = test_h[idx][t].unsqueeze(0).to(device)

                if ablate_M:
                    # Disable temporal accumulation
                    abl_adapter.v_acc = torch.zeros(1, 3, device=device)

                logits, _, _ = abl_adapter(h_t)
                preds.append(logits.argmax(-1).item())
                true.append(test_labels[idx][t].item())

    acc = accuracy_score(true, preds)
    print(f"  {name:35s}: {acc:.1%}")
    return acc

# Get trained state
trained_state = adapter.state_dict()

print("\n📊 Ablation Results:")
acc_full = evaluate_ablation(trained_state, name="Full Model (E+V+M)")
acc_no_E = evaluate_ablation(trained_state, ablate_E=True, name="Without Module E (V+M only)")
acc_no_M = evaluate_ablation(trained_state, ablate_M=True, name="Without Module M (E+V only)")
acc_no_EM = evaluate_ablation(trained_state, ablate_E=True, ablate_M=True, name="Without E+M (V only)")

# Contribution analysis
print(f"\n📊 Module Contribution:")
print(f"  Module E contribution: {(acc_full - acc_no_E)*100:+.1f}pp")
print(f"  Module M contribution: {(acc_full - acc_no_M)*100:+.1f}pp")
print(f"  Module E+M combined:   {(acc_full - acc_no_EM)*100:+.1f}pp")

e_contrib = acc_full - acc_no_E
m_contrib = acc_full - acc_no_M

if abs(e_contrib) > 0.01 or abs(m_contrib) > 0.01:
    print("\n  ✅ Module E and/or M contribute meaningfully!")
    print("  = Dynamic environment activates temporal modules")
    print("  = Confirms PBT hypothesis: E+M need temporal change")
else:
    print("\n  ⚠️ Module E+M contribution minimal")
    print("  = Task may still be too simple for E+M to matter")

# Ablation bar chart
fig, ax = plt.subplots(figsize=(10, 5))
names = ['Full (E+V+M)', 'No E (V+M)', 'No M (E+V)', 'V only']
accs = [acc_full, acc_no_E, acc_no_M, acc_no_EM]
colors = ['#2ecc71', '#e74c3c', '#3498db', '#95a5a6']
bars = ax.bar(names, accs, color=colors, edgecolor='black', lw=1.2)
for b, a in zip(bars, accs):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.005, f'{a:.1%}', ha='center', fontweight='bold')
ax.set_ylim(0, 1.1); ax.set_ylabel('Accuracy'); ax.grid(True, alpha=0.3, axis='y')
ax.set_title('PBT-V v2 Ablation: Module Contribution — Ninthanawat N.', fontweight='bold')
plt.tight_layout(); plt.savefig('pbtv2_ablation.png', dpi=150); plt.show()




  🔬 Ablation Test: Which Modules Matter?

📊 Ablation Results:
  Full Model (E+V+M)                 : 99.8%
  Without Module E (V+M only)        : 99.8%
  Without Module M (E+V only)        : 99.8%
  Without E+M (V only)               : 99.8%

📊 Module Contribution:
  Module E contribution: +0.0pp
  Module M contribution: +0.0pp
  Module E+M combined:   +0.0pp

  ⚠️ Module E+M contribution minimal
  = Task may still be too simple for E+M to matter


In [19]:
# ════════════════════════════════════════════
# CELL 7: Temporal EEG — Anomaly Sequence
# ════════════════════════════════════════════
print("\n" + "=" * 90)
print("  🧠 Temporal EEG: Anomaly Detection in Real-time")
print("=" * 90)

# Find an anomaly sequence in test set
anom_idx = next(i for i, t in enumerate(test_types) if t == "anomaly")

adapter.eval()
adapter.reset_state()

pain_h, pleasure_h, epistemic_h, unsafe_h = [], [], [], []

with torch.no_grad():
    for t in range(SEQ_LEN):
        h_t = test_h[anom_idx][t].unsqueeze(0).to(device)
        logits, v_acc, v_layers, epsilon = adapter(h_t, return_valence=True)

        pain_h.append(v_acc[0, 0].item())
        pleasure_h.append(v_acc[0, 1].item())
        epistemic_h.append(v_acc[0, 2].item())
        unsafe_h.append(F.softmax(logits, dim=-1)[0, 1].item())

# Plot
fig, ax1 = plt.subplots(figsize=(12, 6))
frames = list(range(SEQ_LEN))
ax1.plot(frames, pain_h, 'r-o', lw=2, label='V_pain (Threat)')
ax1.plot(frames, pleasure_h, 'g-s', lw=2, label='V_pleasure (Safe)')
ax1.plot(frames, epistemic_h, 'b-^', lw=2, label='V_epistemic (Surprise)')
ax1.axvspan(3.5, SEQ_LEN-0.5, alpha=0.1, color='red', label='Anomaly Region')
ax1.set_xlabel('Time (Frames)'); ax1.set_ylabel('Accumulated Valence')
ax1.set_xticks(frames)
labels_str = ['Normal']*4 + ['ANOMALY']*4
ax1.set_xticklabels([f'F{i}\n{labels_str[i]}' for i in frames])
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(frames, unsafe_h, 'k--x', lw=2.5, ms=8, label='UNSAFE Prob')
ax2.set_ylabel('UNSAFE Probability'); ax2.set_ylim(-0.05, 1.05)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper left')
plt.title('PBT-V v2 Temporal EEG: Anomaly Detection (CIFAR-10)\nNinthanawat N.', fontweight='bold')
plt.tight_layout(); plt.savefig('pbtv2_eeg_anomaly.png', dpi=150); plt.show()




  🧠 Temporal EEG: Anomaly Detection in Real-time


In [20]:
# ════════════════════════════════════════════
# CELL 8: Temporal EEG — Tricky-Safe
# ════════════════════════════════════════════
print("\n" + "=" * 90)
print("  🧠 Temporal EEG: Tricky-Safe (Should NOT trigger alarm)")
print("=" * 90)

tricky_idx = next(i for i, t in enumerate(test_types) if t == "tricky_safe")

adapter.reset_state()
pain_h2, pleasure_h2, epistemic_h2, unsafe_h2 = [], [], [], []

with torch.no_grad():
    for t in range(SEQ_LEN):
        h_t = test_h[tricky_idx][t].unsqueeze(0).to(device)
        logits, v_acc, v_layers, epsilon = adapter(h_t, return_valence=True)
        pain_h2.append(v_acc[0, 0].item())
        pleasure_h2.append(v_acc[0, 1].item())
        epistemic_h2.append(v_acc[0, 2].item())
        unsafe_h2.append(F.softmax(logits, dim=-1)[0, 1].item())

fig, ax1 = plt.subplots(figsize=(12, 6))
ax1.plot(frames, pain_h2, 'r-o', lw=2, label='V_pain')
ax1.plot(frames, pleasure_h2, 'g-s', lw=2, label='V_pleasure')
ax1.plot(frames, epistemic_h2, 'b-^', lw=2, label='V_epistemic')
ax1.axvline(x=3.5, color='orange', ls='--', lw=2, label='Context Change')
ax1.set_xlabel('Time (Frames)'); ax1.set_ylabel('Accumulated Valence')
labels_str2 = ['Normal']*4 + ['Tricky']*4
ax1.set_xticklabels([f'F{i}\n{labels_str2[i]}' for i in frames])
ax1.set_xticks(frames); ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(frames, unsafe_h2, 'k--x', lw=2.5, ms=8, label='UNSAFE Prob')
ax2.set_ylabel('UNSAFE Probability'); ax2.set_ylim(-0.05, 1.05)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper left')
plt.title('PBT-V v2 Temporal EEG: Tricky-Safe (Should Stay Calm)\nNinthanawat N.', fontweight='bold')
plt.tight_layout(); plt.savefig('pbtv2_eeg_tricky.png', dpi=150); plt.show()

over_refusal = sum(1 for p in unsafe_h2 if p > 0.5)
print(f"\n  Over-refusal: {over_refusal}/{SEQ_LEN} frames triggered (should be 0)")




  🧠 Temporal EEG: Tricky-Safe (Should NOT trigger alarm)

  Over-refusal: 0/8 frames triggered (should be 0)


In [21]:
# ════════════════════════════════════════════
# CELL 9: Module E+M Analysis
# ════════════════════════════════════════════
print("\n" + "=" * 90)
print("  📊 Module E+M: Did They Wake Up?")
print("=" * 90)

# Precision analysis
precs = adapter.get_precision().detach().cpu().numpy()
gammas = adapter.get_gamma().detach().cpu().numpy()

print(f"\n  Module E — Learned Precision Π_l:")
for l in range(N_LAYERS):
    zone = "Perception" if l < 4 else "Evaluation" if l < 8 else "Decision"
    print(f"    Layer {l:2d} ({zone:10s}): Π = {precs[l]:.4f}")

print(f"\n  Module M — Learned Decay Rates:")
print(f"    γ_pain     = {gammas[0]:.4f} (slowest = trauma persists)")
print(f"    γ_pleasure = {gammas[1]:.4f}")
print(f"    γ_epistemic= {gammas[2]:.4f} (fastest = surprise resolves)")
print(f"    Ordering γ_p < γ_pl < γ_e: {'✅' if gammas[0] < gammas[1] < gammas[2] else '❌'}")

# Compare with LLM v6.1
print(f"\n  📊 Comparison: Vision vs LLM")
print(f"    {'Metric':<25s} {'LLM v6.1':>12s} {'Vision v2':>12s}")
print(f"    {'─'*50}")
print(f"    {'Π_max':<25s} {'1.0070':>12s} {precs.max():>12.4f}")
print(f"    {'Π_max - 1.0 (movement)':<25s} {'0.0070':>12s} {precs.max()-1.0:>12.4f}")
print(f"    {'γ_pain change':<25s} {'0.4%':>12s} {abs(gammas[0]-0.0474)/0.0474*100:>11.1f}%")
print(f"    {'Ablation E+M impact':<25s} {'0.1%':>12s} {(acc_full-acc_no_EM)*100:>11.1f}pp")




  📊 Module E+M: Did They Wake Up?

  Module E — Learned Precision Π_l:
    Layer  0 (Perception): Π = 0.9659
    Layer  1 (Perception): Π = 1.1005
    Layer  2 (Perception): Π = 1.5685
    Layer  3 (Perception): Π = 3.0381
    Layer  4 (Evaluation): Π = 2.9003
    Layer  5 (Evaluation): Π = 1.4522
    Layer  6 (Evaluation): Π = 1.2273
    Layer  7 (Evaluation): Π = 1.4277
    Layer  8 (Decision  ): Π = 0.8959
    Layer  9 (Decision  ): Π = 0.8256
    Layer 10 (Decision  ): Π = 1.4817
    Layer 11 (Decision  ): Π = 0.7699

  Module M — Learned Decay Rates:
    γ_pain     = 0.1046 (slowest = trauma persists)
    γ_pleasure = 0.4589
    γ_epistemic= 0.5689 (fastest = surprise resolves)
    Ordering γ_p < γ_pl < γ_e: ✅

  📊 Comparison: Vision vs LLM
    Metric                        LLM v6.1    Vision v2
    ──────────────────────────────────────────────────
    Π_max                           1.0070       3.0381
    Π_max - 1.0 (movement)          0.0070       2.0381
    γ_pain change   

In [22]:
# ════════════════════════════════════════════
# CELL 10: Summary + Download
# ════════════════════════════════════════════
print("\n" + "=" * 90)
print("  🏆 FINAL SUMMARY — PBT-V v2")
print("=" * 90)

print(f"""
┌───────────────────────────────────────────────────────────────────────────┐
│ PBT-V v2: Large-Scale Vision Experiment                                  │
│ DINOv2-Base (768 dim, 12 layers) | CIFAR-10 Temporal Sequences           │
├───────────────────────────────────────────────────────────────────────────┤
│ Dataset: {n_total} sequences × {SEQ_LEN} frames = {n_total*SEQ_LEN} total frames              │
│ Train: {len(train_h)} | Val: {len(val_h)} | Test: {len(test_h)} sequences                     │
│ Types: {N_NORMAL} normal + {N_ANOMALY} anomaly + {N_TRICKY} tricky-safe               │
├───────────────────────────────────────────────────────────────────────────┤
│ Overall Accuracy: {overall_acc:.1%}                                              │
│ Best Val Accuracy: {best_val_acc:.1%}                                            │
├───────────────────────────────────────────────────────────────────────────┤
│ Ablation Results:                                                         │
│   Full (E+V+M): {acc_full:.1%}                                                  │
│   No E (V+M):   {acc_no_E:.1%}  (E contribution: {(acc_full-acc_no_E)*100:+.1f}pp)                  │
│   No M (E+V):   {acc_no_M:.1%}  (M contribution: {(acc_full-acc_no_M)*100:+.1f}pp)                  │
│   V only:       {acc_no_EM:.1%}  (E+M combined: {(acc_full-acc_no_EM)*100:+.1f}pp)                   │
├───────────────────────────────────────────────────────────────────────────┤
│ Module E (Precision): Π_max = {precs.max():.4f}                                  │
│ Module M (Gamma):     γ_p={gammas[0]:.4f} γ_pl={gammas[1]:.4f} γ_e={gammas[2]:.4f}          │
│ γ ordering correct:   {'✅ γ_p < γ_pl < γ_e' if gammas[0]<gammas[1]<gammas[2] else '❌'}                                      │
├───────────────────────────────────────────────────────────────────────────┤
│ PBT Hypothesis Test:                                                      │
│   "Module E+M activate in dynamic/temporal environments"                  │
│   LLM v6.1: E+M impact = 0.1% (static text)                             │
│   Vision v2: E+M impact = {(acc_full-acc_no_EM)*100:.1f}pp (temporal sequences)               │
└───────────────────────────────────────────────────────────────────────────┘
""")

# Download
try:
    from google.colab import files
    files.download("pbt_vision_v2.pth")
    print("📥 Weights downloaded!")
except:
    print("Saved: pbt_vision_v2.pth")



  🏆 FINAL SUMMARY — PBT-V v2

┌───────────────────────────────────────────────────────────────────────────┐
│ PBT-V v2: Large-Scale Vision Experiment                                  │
│ DINOv2-Base (768 dim, 12 layers) | CIFAR-10 Temporal Sequences           │
├───────────────────────────────────────────────────────────────────────────┤
│ Dataset: 2000 sequences × 8 frames = 16000 total frames              │
│ Train: 1400 | Val: 300 | Test: 300 sequences                     │
│ Types: 800 normal + 800 anomaly + 400 tricky-safe               │
├───────────────────────────────────────────────────────────────────────────┤
│ Overall Accuracy: 99.8%                                              │
│ Best Val Accuracy: 99.9%                                            │
├───────────────────────────────────────────────────────────────────────────┤
│ Ablation Results:                                                         │
│   Full (E+V+M): 99.8%                                               

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Weights downloaded!
